# Analysis Notebook for DeepPlateau
## Evaluating Minimal Discs in $\mathbb{H}^4$

In this notebook, we load the trained Physics-Informed Neural Networks (PINNs) generated in the `train_and_save` notebook. With the optimized models loaded, our primary goal is to analyze the geometry of the resulting minimal surfaces; in particular, we locate the double points and compute the self-intersection number.

In [ ]:
import numpy as np

from src.knots import *
from src.extensions import *
from src.full_models import HyperbolicMinimalSurfacePINN
from src.plotting import plot_error, plot_knot, montecarlo_error, plot_mu_heatmap_log
from src.geometry import minimal_in_H4_PDE_flat_new
from src.samplers import MixSampler
from src.double_point_analysis import find_candidate_double_points, refine_double_points_newton
from src.interior_models import MLP

from mpl_toolkits.mplot3d import Axes3D
%matplotlib widget

## 1. Loading the Pre-trained Model

Our first step is to load a fully optimized PINN from the `trained_models/paper` directory. 

By calling the `load` method, we automatically reconstruct the neural network's architecture, its geometric extensions, and the boundary defining functions, before injecting the saved, learned weights. 

In the cell below, we load the minimal surface bounded by a Stevedore knot.

In [ ]:
# load the model
trained_models_path = 'trained_models/paper/'
knot_type = 'stevedore'
knot_parameters = 'R1.6_mirrorFalse'
perturbed = True

model_name = trained_models_path + 'KNOT_' + knot_type + '_KNOT_PAR_' + knot_parameters
if perturbed:
    model_name += '_PERTURBED.pt'
else:
    model_name += '.pt'

model = HyperbolicMinimalSurfacePINN.load(model_name)
untrained_model = HyperbolicMinimalSurfacePINN(**model.kwargs)

## 2. Visualising the boundary knot

To verify that the model and its geometric parameters have been restored correctly, we can extract the boundary knot directly from the loaded model and plot it.

*Note: The 3D plots are interactive. If your notebook environment supports it, you can change the point of view by clicking and dragging the mouse pointer.*

In [ ]:
# show the knot
knot_evaluator = model.get_knot().get_evaluator()
plot_knot(knot_evaluator, dtype=model.kwargs['dtype'])

## 3. Visualising the PDE Residual (Before and After)

In the cell below, we plot the PDE error (the squared norm of the tension field) across the unit disc, of an *untrained* version of our PINN. As expected from our mathematical formulation, the error is identically zero exactly at the boundary—even before any learning occurs—thanks to the stereobiharmonic extension. However, the interior error remains high.

In [ ]:
sampler = MixSampler(dtype=untrained_model.kwargs['dtype'])

# plot the error before training on a regular grid
plot_error(
    minimal_in_H4_PDE_flat_new(untrained_model),
    dtype=untrained_model.kwargs['dtype'],
    vmin=0,
    vmax=1,
    grid_size=300,
    boundary_linewidth=2.6,
    figsize=(6,5),
    colorbar_label='',
    title=None)

Now, we plot the PDE error for our *trained* model. 

By comparing this heat-map to the baseline above, we can visually verify the neural network's work. The interior residual has been driven down uniformly, indicating a successful deformation into a near-minimal p-immersion.

In [ ]:
# plot the error after training on a regular grid
plot_error(
    minimal_in_H4_PDE_flat_new(model),
    dtype=untrained_model.kwargs['dtype'],
    vmin=0,
    vmax=1,
    grid_size=300,
    boundary_linewidth=2.6,
    figsize=(6,5),
    colorbar_label='',
    title=None)

### Monte Carlo Validation

It is important to verify that the loaded model's low PDE residual generalizes across the entire domain. A single grid evaluation provides a good visual sanity check, but a rigorous statistical analysis ensures there are no hidden regions of high error.

In the cell below, we perform a Monte Carlo simulation by drawing 1,000 independent random samples (each containing 16,384 points) from the unit disk. A histogram with a near-zero mean confirms that the surface we restored from the saved weights is near-minimal.

In [ ]:
montecarlo_error(
    minimal_in_H4_PDE_flat_new(model),
    num_samples=1000,
    size_samples=2**14,
    figsize=(5,3),
    bins=100,
    title=None,
    xlabel = 'Loss'
)

## 3. Double Point Analysis

Generic minimal p-immersed surfaces in $\mathbb{H}^4$ intersect themselves transversely at a finite number of double points. Finding and counting these intersections with signs is the key to verifying Fine's Conjecture.

### The Self-Proximity Map

To systematically find these self-intersections, our algorithm works in two phases. The first phase is **candidate generation**. We compute the *self-proximity map* $\mu_\epsilon$, which calculates the minimum spatial distance between $u(p)$ and any other point $u(p')$ on the surface, provided the domain points $p$ and $p'$ are separated by at least a threshold $\epsilon$. The second phase is **candidate refinement**, via Newton's Method.

In the cell below, we plot the logarithm of this self-proximity map. You should look for isolated, concentrated yellow regions — these indicate where the surface is extremely close to itself, revealing the approximate pre-images of the double points.

In [ ]:
_ = plot_mu_heatmap_log(
    # --- Main settings
    u_callable = model,
    epsilon = 0.3,
    cmap='viridis_r',
    grid_resolution=300,
    figsize=(6,5),
    # title = None,
)

### Extracting Candidate Pairs

We now extract specific pairs of points $(p_i, p_j)$ to act as initial guesses for our root-finding algorithm. 

The function below iterates over a regular grid of the domain and filters point pairs based on two critical criteria:
* **Domain threshold ($\epsilon$)**: Ensures the points are sufficiently far apart on the disk, preventing the algorithm from picking trivial points near the diagonal.
* **Codomain threshold ($\tau$)**: Ensures the mapped points $u(p_i)$ and $u(p_j)$ are extremely close to each other in $\mathbb{H}^4$.

The output is a filtered list of candidate pairs, which will typically cluster around the true pre-images of the double points.

In [ ]:
candidates = find_candidate_double_points(
    model,
    grid_resolution=200,
    epsilon=0.2,
    tau=0.03,
    dtype=model.kwargs['dtype'])

print('Number of candidates:', len(candidates))

To verify that our filtering criteria worked as intended, we can overlay the extracted candidate pairs directly onto the self-proximity heatmap. 

In the plot below, the candidates are marked in orange. You should see them tightly clustered around the deepest "valleys" (the yellow regions) of the heatmap, confirming that our initial guesses are correctly localized near the true pre-images of the double points.

In [ ]:
_ = plot_mu_heatmap_log(
    # --- Main settings
    u_callable = model,
    epsilon = 0.3,
    cmap='viridis_r',
    grid_resolution=300,
    figsize=(6,5),
    # title = None,

    # --- Candidate Settings ---
    candidates=candidates,
    candidate_color="#E85D04",
    alpha = 0.8,
)

### Refinement via Newton's Method

With our candidate pairs identified, we proceed to the second phase of the analysis: refining these initial guesses into true double points using Newton's method. 

Because the candidates typically cluster around the same regions, multiple pairs will naturally converge to the identical mathematical root. The function below handles this by automatically deduplicating the converged points.

Crucially, this step also evaluates the mathematical properties of each intersection to verify Fine's Conjecture:
* **The Sign (Jacobian)**: The algorithm computes the determinant of the Jacobian for the difference map $F_u(p_1, p_2) = u(p_1) - u(p_2)$. The sign of this determinant dictates the sign of the double point ($+1$ or $-1$). The sum of these signs gives us the total self-intersection number $d$.
* **Codomain Distance**: As a sanity check, we compute the final Euclidean distance between the mapped points in $\mathbb{H}^4$. For a true double point, this distance should be effectively zero (on the order of $10^{-16}$ due to machine float precision).

In [ ]:
refined_pairs, jacobians, final_distances = refine_double_points_newton(
    model,
    candidates,
    tol=1e-14,
    max_iter=200,
    dedup_tol=1e-11,
    dtype=model.kwargs['dtype'],
)

# Pre-calculate once to improve performance
arr_pairs = np.array(refined_pairs)
arr_jacobians = np.array(jacobians)

# Summaries
pos_mask = arr_jacobians > 0
neg_mask = arr_jacobians <= 0
print(f"Positive double points: {np.sum(pos_mask)}")
print(f"Negative double points: {np.sum(neg_mask)}")
print("-" * 60)

# Header (Updated width for the Dist column to accommodate scientific notation)
print(f"{'#':<5} | {'Type':<10} | {'Jacobian':<10} | {'Codomain Distance':<18} | {'p1 (x,y)':<20} | {'p2 (x,y)':<20}")
print("-" * 96)

for i, (pair, jac, dist) in enumerate(zip(refined_pairs, jacobians, final_distances)):
    pt_type = "Positive" if jac > 0 else "Negative"
    
    # Format coordinates
    p1 = f"({pair[0][0]:.3f}, {pair[0][1]:.3f})"
    p2 = f"({pair[1][0]:.3f}, {pair[1][1]:.3f})"
    
    # dist:<18.2e aligns with the 18-character width of the new header
    print(f"{i+1:<5} | {pt_type:<10} | {jac:<10.3f} | {dist:<18.2e} | {p1:<20} | {p2:<20}")

### Visualizing the Confirmed Double Points

Now that we have rigorously confirmed and classified our double points, we can visualize their exact locations on the domain.

In the cell below, we overlay the refined point pairs onto the self-proximity heatmap. To make the topological properties clear, the plotting function visually distinguishes the intersections based on their computed signs:
* **Positive double points** are marked in yellow.
* **Negative double points** are marked in red.

By counting these points and summing their signs, we arrive at the total self-intersection number for the minimal surface. You can now compare this final count directly with the corresponding coefficient in the boundary knot's HOMFLY polynomial to see Fine's Conjecture in action!

In [ ]:
_ = plot_mu_heatmap_log(
    # --- Main settings
    u_callable = model,
    epsilon = 0.3,
    cmap='viridis_r',
    grid_resolution=300,
    figsize=(6,5),
    # title = None,

    # # --- Candidate Settings ---
    # candidates=candidates,
    # candidate_color="#E85D04",
    # alpha = 0.8,

    # --- Refined pairs Settings ---
    refined_pairs=refined_pairs,
    jacobians=jacobians,
    refined_pos_color='#ffc000',
    refined_neg_color="#fe0c0c",

    # --- Legend Settings ---
    # legend_title = 'Double points',
    legend_label_candidates='Candidate double points',
    legend_label_refined_pos=r'Positive double points',
    legend_label_refined_neg=r'Negative double points',
    # legend_bbox_to_anchor=None,
    # legend_loc='upper right',
    legend_frameon=True,

    # --- Boundary Settings ---
    boundary_color='black', 
    boundary_linewidth=2.6,

    # --- Colorbar Settings ---
    show_colorbar=True,
    colorbar_label=None,
)